# Final Submission
EECS 6412 - Final Project <br>
Module: [Programs] <br>
Authors: Omkumar M. Patel, Rashid Rasooli, Michael Murphy <br>
Date: 21 Dec 2025 <br>
Purpose: Implementing Solution for JSSP Problems using LLMs for Optimization

# Setup
Split into two cells, as Colab needs to restart after installing requirements from Starjob repo.
- Downloads the Starjob codebase from GitHub: https://github.com/starjob42/Starjob.
- Installs pip requirements.
- Downloads the Starjob dataset from HuggingFace, moves to the expected folder for the finetuning workflow.

In [ ]:
!git clone https://github.com/starjob42/Starjob.git

In [ ]:
# Seems to conflict with current requirements of unsloth
#!pip install -r Starjob/requirements.txt

#So manually installing dependencies from Starjob requirements.txt
#Install Unsloth for optimized 4-bit fine-tuning
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install torch==2.9.0
!pip install torchaudio==2.9.0 torchvision==0.24.0

# Install Graph Processing Dependencies
!pip install wandb datasets torch_geometric

In [ ]:
#Make data directory and download Starjob dataset
!mkdir data/
!hf download mideavalwisard/Starjob --repo-type dataset --local-dir data/

We faced issues regarding dependencies as unsloth's new version was released during our training pipeline. That led us into spiral where we kept matching versions and then figured out that unsloth and unsloth-zoo libraries had been updated just when were about to train our model.

# Fine-tuning

## Initial Configuration

- Modifying to use static variables instead of `argparse` (which doesn't work with Colab).
- Also changed `dtype` to `float16`, since the Tesla T4 does not support `bfloat16`.
- Changed back `bfloat16` while running it on NVIDIA A6000 GPU (runpod)

#### Run below Import only when GPU unavailable (i.e., dataset work only)

In [ ]:
# import torch
# from datasets import load_dataset

# import re
# import numpy as np
# from typing import Tuple, List, Dict

# import networkx as nx
# from collections import defaultdict

# from torch_geometric.data import Data
# from torch_geometric.utils import from_networkx
# from tqdm import tqdm

#### When GPU available

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
from datasets import load_dataset,Dataset
import argparse
import torch
import wandb
import re
import numpy as np
from typing import Tuple, List, Dict
import networkx as nx
from collections import defaultdict
from tqdm import tqdm
from torch_geometric.data import Data
from torch_geometric.utils import from_networkx
from trl import SFTTrainer
from transformers import TrainingArguments

NOTE: Original script uses argparse, which doesn't seem to work with Colab. Initially we ran our code on colab (T4 GPU) and later moved to a runpod setup with NVIDIA A6000 GPU. Thus, Converted to plain variables instead


In [ ]:
## Model and data parameters
max_seq_length = 4096
# max_seq_length = 50000
dtype = 'bfloat16'           
load_in_4bit = True

# LoRA hyperparameters
lora_r = 64
lora_alpha = 64
lora_dropout = 0.0
bias = 'none'

# Additional configurations
use_gradient_checkpointing = 'unsloth'
random_state = 42
use_rslora = False
loftq_config = None

# Training hyperparameters
per_device_train_batch_size = 4
gradient_accumulation_steps = 4
warmup_steps = 5
num_train_epochs = 2
learning_rate = 2e-4
logging_steps = 1
optim = 'adamw_8bit'
weight_decay = 0.01
lr_scheduler_type = 'linear'
seed = 42
save_total_limit = 50
save_step = 200
per_device_eval_batch_size = 2
train_lm_head = False
train_embed_tokens = False

# Output directory
output_dir = None

In [ ]:
# Create an output directory name based on hyperparameters
if output_dir is None:
    dir_out = f"output_alpha{lora_alpha}_r{lora_r}_train_lm_head{train_lm_head}_train_embed_tok_{train_embed_tokens}_seq{max_seq_length}_b{per_device_train_batch_size}_ep{num_train_epochs}"
else:
    dir_out = output_dir
    
# Initialize Weights & Biases for experiment tracking
wandb.init(
    project="Project-Name",  # Change the project name as per your wandb setup
    name=dir_out,
)

## Initialising model

In [ ]:
# Load Model and Tokenizer
dtype = torch.bfloat16 if dtype == 'bfloat16' else torch.float16 # Set dtype based on availability of GPU support

# Load the pre-trained model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

target_modules =[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ]
if train_lm_head:
    target_modules.append('lm_head')
if train_embed_tokens:
    target_modules.append('embed_tokens')

# Configure the model with PEFT (Parameter-Efficient Fine-Tuning)
model = FastLanguageModel.get_peft_model(
    model,
    r=lora_r,
    target_modules=target_modules,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    bias=bias,
    use_gradient_checkpointing=use_gradient_checkpointing,
    random_state=random_state,
    use_rslora=use_rslora,
    loftq_config=loftq_config,
)

In [ ]:
# Define the Alpaca-style prompt template
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""
EOS_TOKEN = tokenizer.eos_token
# EOS_TOKEN = "END"       # Uncomment this only when GPU unavailable

## Preparing dataset
- Moved the `alpaca_prompt` template string and `formatting_prompts_func` function into this area, for cleanliness.
- Added extra comments for clarity.
- Incorporating the idea from Om's notebook, i.e., removing old-style Starjob instances: https://colab.research.google.com/drive/1bZjfnDFPrsp3CBXNa2nXr0AJtKXLDlgw?usp=sharing

In [ ]:
# Load and Prepare Dataset
dataset = load_dataset('./data/', split="train") # load the data from the ./data folder
dataset_new_style = dataset.filter(lambda x: x['matrix'] is not None) # Remove old-style instances
print(f"lenght of og dataset: {len(dataset)} \n lenght of new-style dataset: {len(dataset_new_style)}")

# For Colab (T4 GPU) friendly Processing, we xperimented with only 20% of the new dataset.
# Obtain train/test splits
split_dataset = dataset_new_style.train_test_split(test_size=0.02, seed=seed)

# function to create prompts with template, using columns from the dataset
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input_text, output) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

# Map the train/test splits through the above function, to create prompts for the LLM
train_dataset = split_dataset['train'].map(formatting_prompts_func, batched=True)
eval_dataset = split_dataset['test'].map(formatting_prompts_func, batched=True)
print(f"Number of train examples: {len(train_dataset)} \n\n Number of test examples: {len(eval_dataset)}")

### Issue with dataset
NOTE: Seems to be a significant issue with this script. We can see in ![Report: Figure 4](https://docs.google.com/document/d/1tWRiAyn0QzmGB23Htouj9FCSvQbbggZP4gwJh7FfjXM/edit?usp=sharing) that the "text" column of the old-style instances (i.e., from the original "LLMs can Schedule" paper) has _no_ instruction or input in its "text" column, only a response. This means that the LLM is being trained on useless prompts.

After dataset evaluation, we print the not null dataset


In [ ]:
# New-style: Has a "matrix" column
new_sample = None
i = 0
while True:
    i += 1
    if train_dataset[i]['matrix'] is not None:
        new_sample = train_dataset[i]
        # print(new_sample)         # All columns
        print(new_sample["text"]) # Only the "text" column (i.e., the one used by the LLM)
        break

## Creating GNN Dataset

### GNN Encoding of Instances

#### Functions

In [ ]:
# Convert "new-style" Dataset to GNN Encoding
# Function to parse JSSP instances (from Starjob paper) into structured format.
def parse_jssp_instance(instance: Dict) -> Dict:
    # Number of jobs and machines are already stored in the instance.
    # We need to parse operations
    job_operations = {}
    lines = instance['input'].strip().split('\n')
    for i in range(len(lines)):
        # Assumes 'jobs' specified every second line, with 'machine-operations' every other line
        if i % 2 == 0:
            job_id = int(re.search(r'J(\d+):', lines[i]).group(1))
            operations = re.findall(r'M(\d+):(\d+)', lines[i+1])
            job_operations[job_id] = [(int(m), int(d)) for m, d in operations]
        # Skip every other line, as we already handle them above
        else:
            continue

    # Also extract number of operations
    # NOTE: We assume that each job has the same number of operations, so we just multiply
    # the number of operations in the first job by the number of jobs
    num_operations = instance['num_jobs'] * len(next(iter(job_operations.values())))

    # Return everything as structured dictionary
    return {
        'num_jobs': instance['num_jobs'],
        'num_machines': instance['num_machines'],
        'num_operations': num_operations,
        'job_operations': job_operations,
    }

In [ ]:
# Function to build disjunctive graph representation of the parsed (structured) JSSP dictionary
# Uses networkx.DiGraph to store the directed graph, for better interpretability
def build_disjunctive_graph(parsed_problem: Dict) -> Tuple[nx.DiGraph, Dict]:
    # Initialise the graph
    G = nx.DiGraph()

    # Extract the operations for this JSSP instance
    job_ops = parsed_problem['job_operations']

    # Add source and sink nodes
    source = ('source', 0)
    sink = ('sink', 0)
    G.add_node(source, node_type='source', processing_time=0)
    G.add_node(sink, node_type='sink', processing_time=0)

    # Initialise reference dictionariesMap operation nodes and create node info dictionary
    node_to_operation = {}  # node -> (job_id, op_id, machine_id, proc_time) [NOT NEEDED?]
    # operation_to_node = {}  # (job_id, op_id) -> node [NOT NEEDED?]
    operations_by_machine = defaultdict(list)   # machine_id -> [nodes]

    # Iterate over each job to create nodes of each operation
    for job_id, operations in job_ops.items():
        # Then iterate over each operation of that job
        for op_id, (machine_id, proc_time) in enumerate(operations):
            # Each node stores the current job and operation
            node = (job_id, op_id)

            # Add the node, with relevant info for this operation
            G.add_node(node,
                      node_type='operation',
                      job_id=job_id,
                      operation_id=op_id,
                      machine_id=machine_id,
                      processing_time=proc_time)

            # Add same info to our reference dictionaries
            node_to_operation[node] = (job_id, op_id, machine_id, proc_time)
            # operation_to_node[(job_id, op_id)] = node
            operations_by_machine[machine_id].append(node)

    # Iterate over each job to add conjunctive edges (precedence constraints within jobs)
    for job_id, operations in job_ops.items():
        # First edge: Source → first operation of this job
        # first_op = operation_to_node[(job_id, 0)]
        first_op = (job_id, 0)
        G.add_edge(source, first_op, edge_type='conjunctive', weight=0)

        # Main edges: Operations in sequence within this job
        for op_id in range(len(operations) - 1):
            # from_node = operation_to_node[(job_id, op_id)]
            # to_node = operation_to_node[(job_id, op_id + 1)]
            from_node = (job_id, op_id)
            to_node = (job_id, op_id + 1)
            proc_time = operations[op_id][1]
            G.add_edge(from_node, to_node, edge_type='conjunctive', weight=proc_time)

        # Last edge: Last operation of this job -> sink
        # last_op = operation_to_node[(job_id, len(operations) - 1)]
        last_op = (job_id, len(operations) - 1)
        last_proc_time = operations[-1][1]
        G.add_edge(last_op, sink, edge_type='conjunctive', weight=last_proc_time)

    # Iterate over each machine to add disjunctive edges (machine constraints)
    # These will be undirected initially; orientation and weighting happens during scheduling
    for machine_id, ops_on_machine in operations_by_machine.items():
        # Initial graph is complete bipartite graph (all pairs can conflict)
        for i in range(len(ops_on_machine)):
            for j in range(i + 1, len(ops_on_machine)):
                op1, op2 = ops_on_machine[i], ops_on_machine[j]
                # Add as undirected edge (will be oriented during solution)
                G.add_edge(op1, op2, edge_type='disjunctive_undirected', machine_id=machine_id)
                G.add_edge(op2, op1, edge_type='disjunctive_undirected', machine_id=machine_id)

    # Return initialised graph, with mappings of each node to its associated operation info
    return G, node_to_operation

In [ ]:
# Function to convert networkx.DiGraph to a PyTorch Geometric Data object
def graph_to_pygdata(G: nx.DiGraph, node_to_operation: Dict) -> Data:
    
    # NOTE: Not currently using node_to_operation, as all the op info is stored in the node
    # Extract all operation nodes (exclude source/sink initially)
    operation_nodes = [n for n in G.nodes() if G.nodes[n]['node_type'] == 'operation']
    num_jobs = max([G.nodes[n]['job_id'] for n in operation_nodes]) + 1
    num_machines = max([G.nodes[n]['machine_id'] for n in operation_nodes]) + 1
    max_ops_per_job = max([G.nodes[n]['operation_id'] for n in operation_nodes]) + 1

    # Create node index mapping (including source/sink nodes)
    # NOTE: Reordering the list to put source at start and sink at end, but this may not be needed
    # If so, can just use `node_idx = {node: i for i, node in enumerate(list(G))}`
    node_list = [list(G)[0]] + operation_nodes + [list(G)[1]]
    node_idx = {node: i for i, node in enumerate(node_list)}

    # Iterate over nodes to create node feature matrix
    num_nodes = len(node_list)
    node_features = []
    for node in node_list:
        if G.nodes[node]['node_type'] == 'source':
            # Source node: all zeros
            features = [0] * (num_machines + num_jobs + 2)
            node_features.append(features)
        elif G.nodes[node]['node_type'] == 'sink':
            # Sink node: all zeros
            features = [0] * (num_machines + num_jobs + 2)
            node_features.append(features)
        else:
            # Operation node
            machine_id = G.nodes[node]['machine_id']
            job_id = G.nodes[node]['job_id']
            op_idx = G.nodes[node]['operation_id']
            proc_time = G.nodes[node]['processing_time']

            # One-hot machine encoding
            machine_one_hot = [1 if i == machine_id else 0 for i in range(num_machines)]

            # One-hot job encoding
            job_one_hot = [1 if i == job_id else 0 for i in range(num_jobs)]

            # Normalized operation index and processing time
            op_idx_norm = op_idx / max(1, max_ops_per_job - 1)
            proc_time_norm = proc_time / 500.0  # Normalize by max typical processing time

            features = machine_one_hot + job_one_hot + [op_idx_norm, proc_time_norm]
            node_features.append(features)

    x = torch.tensor(node_features, dtype=torch.float32)

    # Create edge list
    edge_index = [[], []]
    edge_attr = []

    for u, v, data in G.edges(data=True):
        u_idx = node_idx[u]
        v_idx = node_idx[v]
        edge_index[0].append(u_idx)
        edge_index[1].append(v_idx)

        # Edge type: conjunctive=1, disjunctive=0
        edge_type = 1 if data['edge_type'] == 'conjunctive' else 0
        weight = data.get('weight', 0) / 500.0  # Normalize weight

        edge_attr.append([edge_type, weight])

    edge_index = torch.tensor(edge_index, dtype=torch.long)
    edge_attr = torch.tensor(edge_attr, dtype=torch.float32) if edge_attr else torch.zeros((0, 2))

    # Create PyTorch Geometric Data object
    pyg_data = Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr,
        num_nodes=num_nodes,
        num_jobs=num_jobs,
        num_machines=num_machines,
        node_mapping=node_idx
    )

    return pyg_data

In [ ]:
# Use above utility functions to convert natural language grompt into PyG Data
from pprint import pp

# def convert_starjob_prompt_to_gnn(nl_prompt: str) -> Data:
def convert_starjob_instance_to_gnn(instance: Dict) -> Data:
    # Step 1: Parse
    parsed = parse_jssp_instance(instance)

    # Step 2: Build graph
    G, node_to_op = build_disjunctive_graph(parsed)
    
    # Step 3: Convert to PyG
    pyg_data = graph_to_pygdata(G, node_to_op)
    
    return pyg_data

pyg_data = convert_starjob_instance_to_gnn(new_sample)

In [ ]:
print(f"Nodes: {pyg_data.num_nodes}")
print(f"Edges: {pyg_data.edge_index.shape[1]}")
print(f"Node features shape: {pyg_data.x.shape}")
print(f"Edge features shape: {pyg_data.edge_attr.shape}")

#### Encoding full datasets and saving them

In [ ]:
# Convert all prompts to GNN representations
train_dataset_gnn = []
for instance in tqdm(train_dataset):
    try:
        pyg_data = convert_starjob_instance_to_gnn(instance)
        pyg_data.target_solution = instance['output']  # Attach solution
        train_dataset_gnn.append(pyg_data)
    except Exception as e:
        # print(f"Failed to parse: {e}")
        continue
print(f"Created {len(train_dataset_gnn)} GNN instances")

In [ ]:
torch.save(train_dataset_gnn, "gnn_dataset/train_dataset_gnn.pt")

In [ ]:
# Same on eval_dataset
eval_dataset_gnn = []
for instance in tqdm(eval_dataset):
    try:
        pyg_data = convert_starjob_instance_to_gnn(instance)
        pyg_data.target_solution = instance['output']
        eval_dataset_gnn.append(pyg_data)
    except Exception as e:
        continue

In [ ]:
torch.save(eval_dataset_gnn, "gnn_dataset/eval_dataset_gnn.pt")

### Reload datasets if necessary

In [ ]:
train_dataset_gnn = torch.load("gnn_dataset/train_dataset_gnn.pt", weights_only=False)
eval_dataset_gnn = torch.load("gnn_dataset/eval_dataset_gnn.pt", weights_only=False)

### Serializing GNN-based instances for LLM fine-tuning

In [ ]:
def serialize_pyg_constraints_rules(pyg_data: Data, original_instance: Dict = None) -> str:
    num_jobs = pyg_data.num_jobs
    num_machines = pyg_data.num_machines
    
    # Extract structure
    nodes_by_job = defaultdict(list)
    operations_by_machine = defaultdict(list)
    node_features_map = {}
    
    for node_idx in range(1, pyg_data.num_nodes - 1):
        x_features = pyg_data.x[node_idx]
        machine_id = torch.argmax(x_features[:num_machines]).item()
        job_id = torch.argmax(x_features[num_machines:num_machines+num_jobs]).item()
        
        op_idx = int(x_features[-2].item() * 10)
        proc_time = int(x_features[-1].item() * 500)
        
        nodes_by_job[job_id].append((op_idx, node_idx, machine_id, proc_time))
        operations_by_machine[machine_id].append((job_id, op_idx, node_idx, proc_time))
        node_features_map[node_idx] = (job_id, op_idx, machine_id, proc_time)
    
    lines = ["## CONSTRAINT SPECIFICATION FOR JSSP\n"]
    
    # 1. Job precedence constraints (ALL OF THEM)
    lines.append("### JOB PRECEDENCE CONSTRAINTS")
    lines.append("(Each job's operations must execute in fixed order)\n")

    precedence_seen = set()  # Deduplication

    for job_id in sorted(nodes_by_job.keys()):
        ops = sorted(nodes_by_job[job_id], key=lambda x: x[0])
        op_names = [f"J{job_id}_Op{op_idx}" for op_idx, _, _, _ in ops]

        if len(op_names) == 0:
            continue

        # Single chain line for readability
        chain_str = " -> ".join(op_names)
        lines.append(f"Job {job_id}: {chain_str}")

        # If hardware is strong enough, also emit full explicit precedence rules
        # for i in range(len(op_names) - 1):
        #     constraint_pair = (op_names[i], op_names[i+1])
        #     if constraint_pair not in precedence_seen:
        #         lines.append(f"MUST_PRECEDE: {op_names[i]} -> {op_names[i+1]}")
        #         lines.append(f"  Constraint: {op_names[i]} must complete before {op_names[i+1]} starts")
        #         precedence_seen.add(constraint_pair)
        lines.append("")  # Blank line between jobs
    
    # 2. Machine exclusivity constraints (ALL OF THEM)
    lines.append("\n### MACHINE EXCLUSIVITY CONSTRAINTS")
    lines.append("(Operations on same machine cannot overlap)\n")

    mutex_seen = set()  # Deduplication
    
    for machine_id in sorted(operations_by_machine.keys()):
        # ops = operations_by_machine[machine_id]
        ops = sorted(operations_by_machine[machine_id], key=lambda x: (x[0], x[1]))  # Sort by job, op_idx
        op_names = [f"J{job_id}_Op{op_idx}" for job_id, op_idx, _, _ in ops]

        if len(op_names) == 0:
            continue
        
        lines.append(f"MACHINE {machine_id} OPERATIONS: {{{', '.join(op_names)}}}")
        
        # If hardware is strong enough, also generate all pairwise exclusivity constraints
        # for i in range(len(ops)):
        #     for j in range(i + 1, len(ops)):
        #         op1_name = op_names[i]
        #         op2_name = op_names[j]

        #         constraint_pair = tuple(sorted([op1_name, op2_name]))
        #         if constraint_pair not in mutex_seen:
        #             lines.append(f"  MUTEX: {op1_name} (+) {op2_name}")
        #             mutex_seen.add(constraint_pair)
        # Else, put in a generic instruction to obey exclusivity constraints
        lines.append(
            "  Rule: Operations in this set must NOT execute in parallel; "
            "they must be totally ordered in time."
        )
        
        lines.append("")
    
    # 3. Schedule feasibility rules
    lines.append("\n### SCHEDULING RULES")
    lines.append("To generate a valid schedule, you MUST:")
    lines.append("  1. Respect ALL precedence constraints ( -> )")
    lines.append("  2. Ensure NO machine has overlapping operations ( (+) )")
    lines.append("  3. Assign start times to each operation based on these constraints")
    lines.append("  4. Minimize the maximum end time (makespan)\n")
    
    # 4. Summary statistics
    lines.append("### CONSTRAINT SUMMARY")
    # total_precedence = sum(len(ops) - 1 for ops in nodes_by_job.values()) + num_jobs
    total_precedence = sum(len(ops) - 1 for ops in nodes_by_job.values())
    total_mutex = sum(len(ops) * (len(ops) - 1) // 2 for ops in operations_by_machine.values())
    lines.append(f"Total precedence constraints: {total_precedence}")
    lines.append(f"Total machine exclusivity constraints: {total_mutex}")
    lines.append(f"Total constraints: {total_precedence + total_mutex}\n")
    
    return "\n".join(lines)

In [ ]:
# def convert_gnn_dataset_to_augmented_format(gnn_dataset, train_dataset_orig, 
def convert_gnn_dataset_to_augmented_format(gnn_dataset, 
                                            tokenizer, alpaca_prompt):
    texts = []
    
    for idx, gnn_item in tqdm(enumerate(gnn_dataset)):
        # instruction = train_dataset_orig[idx]['instruction']
        instruction = f"Optimize schedule for {gnn_item.num_jobs} Jobs (denoted as J) across {gnn_item.num_machines} Machines (denoted as M) to minimize makespan. The makespan is the completion time of the last operation in the schedule. Each M can process only one J at a time, and once started, J cannot be interrupted."
        solution = gnn_item.target_solution
        
        # Serialize graph using chosen method
        graph_representation = serialize_pyg_constraints_rules(gnn_item)
        # graph_representation = serialize_pyg_constraints_rules(gnn_item, train_dataset_orig[idx] if idx < len(train_dataset_orig) else {})
        
        formatted = alpaca_prompt.format(instruction, graph_representation, solution) + EOS_TOKEN
        texts.append(formatted)
    
    return Dataset.from_dict({'text': texts, 'idx': list(range(len(texts)))})

In [ ]:
print("Converting GNN datasets to text format for LLM training...")
# Create augmented text datasets
train_dataset_gnn_text = convert_gnn_dataset_to_augmented_format(
    gnn_dataset=train_dataset_gnn,
    # train_dataset_orig=train_dataset,
    tokenizer=tokenizer,
    alpaca_prompt=alpaca_prompt,
)

eval_dataset_gnn_text = convert_gnn_dataset_to_augmented_format(
    gnn_dataset=eval_dataset_gnn,
    # train_dataset_orig=eval_dataset,
    tokenizer=tokenizer,
    alpaca_prompt=alpaca_prompt,
)

print(f"Training dataset: {len(train_dataset_gnn_text)} instances")
print(f"Eval dataset: {len(eval_dataset_gnn_text)} instances")

# Verify format
print("\nSample training instance:")
print(train_dataset_gnn_text[0]['text'][:500])  # Print first 500 chars

In [ ]:
#Saving the converted datasets
train_dataset_gnn_text.to_json(
    "gnn_dataset/train_dataset_gnn_text.json"
)
eval_dataset_gnn_text.to_json(
    "gnn_dataset/eval_dataset_gnn_text.json"
)

## Training (with GNN)

### Reloading Loading the Fully processed Dataset

In [ ]:
train_dataset_gnn_text = load_dataset(
    "json",
    data_files="/content/drive/MyDrive/EECS 6412 - Project/3. Solution & Final Report/GNN_DATASET/train_dataset_gnn_text.json",
    split="train",
)
eval_dataset_gnn_text = load_dataset(
    "json",
    data_files="/content/drive/MyDrive/EECS 6412 - Project/3. Solution & Final Report/GNN_DATASET/eval_dataset_gnn_text.json",
    split="train",
)


### Normal Training (Tried on Colab)

In [ ]:
# Initialize the Trainer with GNN-augmented dataset
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset_gnn_text,  # Using GNN-derived datasets
    eval_dataset=eval_dataset_gnn_text,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    # dataset_num_proc=20, #Got runtime error as unsloth tries to tokenize parts of dataset using multi-processing
    dataset_num_proc=1,
    packing=False,

    args=TrainingArguments(
        per_device_train_batch_size=per_device_train_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        warmup_steps=warmup_steps,
        num_train_epochs=num_train_epochs,
        learning_rate=learning_rate,
        # fp16=True,
        bf16=is_bfloat16_supported(),
        logging_steps=logging_steps,
        optim=optim,
        weight_decay=weight_decay,
        lr_scheduler_type=lr_scheduler_type,
        seed=seed,
        output_dir=dir_out,
        # report_to="wandb",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=save_total_limit,
        save_steps=save_step,
        eval_strategy="steps",
        eval_steps=save_step,
        per_device_eval_batch_size=per_device_eval_batch_size,
    ),
)

### Pre-tokenising before training

In [ ]:
#Trying to pre-tokenize with single process coz unsloth is still ignoring dataset_num_proc=1
def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        max_length=max_seq_length,
        truncation=True,
        padding=False,
    )

tokenized_train = train_dataset_gnn_text.map(
    tokenize_function,
    batched=True,
    num_proc=1,              # <- single process
    remove_columns=train_dataset_gnn_text.column_names,
)

tokenized_eval = eval_dataset_gnn_text.map(
    tokenize_function,
    batched=True,
    num_proc=1,
    remove_columns=eval_dataset_gnn_text.column_names,
)


Now tokenized_train / tokenized_eval contain input_ids and attention_mask already, with no further multiprocessing

In [ ]:
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

training_args = TrainingArguments(
    output_dir=dir_out,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_eval_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    num_train_epochs=num_train_epochs,
    learning_rate=learning_rate,
    logging_steps=logging_steps,
    bf16=is_bfloat16_supported(),
    optim=optim,
    weight_decay=weight_decay,
    lr_scheduler_type=lr_scheduler_type,
    warmup_steps=warmup_steps,
    seed=seed,
    save_total_limit=save_total_limit,    
    save_steps=save_step,
    eval_strategy="steps",
    eval_steps=save_step,
    report_to="wandb",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
)


### Fine-tuning and saving model

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0) # usage stats
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

# Start Training
trainer_stats = trainer.train()

In [ ]:
model_dir = "model/"
print(f"starting to model components to {model_dir}")

model.save_pretrained(model_dir)
print(f"Saved pre-trained model to {model_dir}")

In [ ]:
# saving tokenizer separately
tokenizer_dir = model_dir + "/tokenizer"
tokenizer.save_pretrained(tokenizer_dir)
print(f"Saved tokenizer to {tokenizer_dir}")

In [ ]:
# saving trained model separately
trained_dir = model_dir + "/trained_model"
trainer.save_model(trained_dir)
print(f"Saved fine-tuned model to {trained_dir}")